# Pipeline Enrich Bằng Overpass API Cho Bộ 100 Listing

Notebook này enrich bộ dữ liệu `100` nhà riêng ở Gò Vấp và Tân Bình bằng `Overpass API` của OpenStreetMap và lưu kết quả trong thư mục `data/overpass/`.

Điểm khác với bản cũ:
- không hardcode POI thủ công
- gọi API thật theo `latitude/longitude` của từng BĐS
- có cache để tránh gọi lại
- sinh feature có thể tái sử dụng cho ranking và validation

Ghi chú:
- Dataset đã có `latitude` và `longitude`, nên không cần geocoding.
- Overpass là public API, cần hạn chế tần suất gọi và bật cache.
- Theo tài liệu chính thức của Overpass, không có một mức `request/giây` cố định cho mọi lúc. Quota phụ thuộc vào tải hệ thống, số slot khả dụng, thời gian chạy truy vấn và thời gian cool down.
- Tài liệu cũng nêu một ngưỡng an toàn khá rộng là khoảng `10.000 request/ngày` và dưới `1 GB/ngày`, nhưng đây không phải cam kết rằng client có thể bắn liên tục mà không bị `429`.
- Vì vậy notebook này được chỉnh theo hướng: kiểm tra `api/status`, chờ slot, chạy theo batch, cache kết quả và chịu lỗi tốt hơn.
- Nếu sau này muốn chuyển sang Google Places hay Geoapify, có thể giữ nguyên schema output.

## Chạy Nhanh

Notebook này dành cho pipeline `Overpass API` hiện tại của nhóm.

Input chính:
- `data/go_vap_tan_binh_100.json`

Output chính:
- `data/overpass/go_vap_tan_binh_100_enriched_overpass_api.json`
- `data/overpass/go_vap_tan_binh_100_enriched_overpass_api_checkpoint.json`
- `data/overpass/go_vap_tan_binh_100_overpass_errors.json`
- `data/overpass/cache/`

Tài liệu liên quan:
- `data/overpass/overpass_enriched_schema_readme.json`
- `data/overpass/overpass_api_curl_examples.md`
- `docs/provider_comparison_overpass_vs_geoapify.md`

Lưu ý:
- Đây là bản notebook API chính cho Overpass.
- Không dùng notebook `notebooks/enrich_gv_tb_100.ipynb` nếu mục tiêu là enrich bằng API thật.


## Bước 0. Khởi Tạo Đường Dẫn Và Thư Mục Cache

Mục đích:
- Xác định đường dẫn gốc của project.
- Khai báo file input, file output và thư mục cache.
- Khai báo endpoint của Overpass API.

Input:
- Cấu trúc thư mục hiện tại của repo.

Output:
- Các biến `DATA_DIR`, `OVERPASS_DIR`, `CACHE_DIR`, `INPUT_PATH`, `OUTPUT_PATH`, `OVERPASS_URL`, `USER_AGENT`.
- Thư mục `data/overpass/` và thư mục cache bên trong được tạo nếu chưa tồn tại.


In [1]:
from pathlib import Path
import hashlib
import json
import math
import re
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data'
OVERPASS_DIR = DATA_DIR / 'overpass'
OVERPASS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OVERPASS_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / 'go_vap_tan_binh_100.json'
OUTPUT_PATH = OVERPASS_DIR / 'go_vap_tan_binh_100_enriched_overpass_api.json'
CHECKPOINT_PATH = OVERPASS_DIR / 'go_vap_tan_binh_100_enriched_overpass_api_checkpoint.json'
ERROR_LOG_PATH = OVERPASS_DIR / 'go_vap_tan_binh_100_overpass_errors.json'

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
OVERPASS_STATUS_URL = 'https://overpass-api.de/api/status'
USER_AGENT = 'IT2041_G8_DecisionMaking/1.0 (overpass notebook)'

INPUT_PATH, OUTPUT_PATH, CACHE_DIR

(PosixPath('/Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/go_vap_tan_binh_100.json'),
 PosixPath('/Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/go_vap_tan_binh_100_enriched_overpass_api.json'),
 PosixPath('/Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/cache/overpass_enrichment'))

## Bước 0.1. Cấu Hình Chạy

Mục đích:
- Thiết lập bán kính tìm kiếm, bán kính đếm POI, thời gian chờ giữa các request và số lần retry.
- Chốt danh sách category cần enrich.

Input:
- Các tham số cấu hình do người dùng điều chỉnh.

Output:
- Bộ cấu hình dùng xuyên suốt notebook, ví dụ `MAX_RADIUS_M`, `COUNT_RADIUS_M`, `REQUEST_SLEEP_SECONDS`, `CATEGORY_ORDER`.


In [2]:
MAX_RADIUS_M = 1500
COUNT_RADIUS_M = 1000
BASE_SLEEP_SECONDS = 2.0
STATUS_POLL_SECONDS = 15
RETRY_COUNT = 5
TIMEOUT_SECONDS = 60

START_INDEX = 0
TARGET_TOTAL_PROPERTIES = 100
MAX_PROPERTIES_PER_RUN = 15
BATCH_PAUSE_SECONDS = 30
RUN_UNTIL_DONE = True
MAX_BATCHES = 30
CONTINUE_ON_ERROR = True
FORCE_REFRESH = False
SAVE_CHECKPOINT_EVERY = 5

CATEGORY_ORDER = [
    'school',
    'park',
    'hospital',
    'supermarket',
    'market',
    'cafe',
    'boulevard',
]

CATEGORY_ORDER

['school', 'park', 'hospital', 'supermarket', 'market', 'cafe', 'boulevard']

## Bước 0.2. Nạp Dữ Liệu Gốc

Mục đích:
- Đọc bộ dữ liệu 100 bất động sản để chuẩn bị enrich.
- Kiểm tra nhanh quy mô và một vài cột quan trọng.

Input:
- File `data/go_vap_tan_binh_100.json`.

Output:
- Biến `properties` dạng danh sách JSON.
- `DataFrame` tạm thời để preview dữ liệu đầu vào.


In [3]:
with INPUT_PATH.open('r', encoding='utf-8') as f:
    properties = json.load(f)

df = pd.DataFrame(properties)
print('Số bản ghi:', len(properties))
display(df.head(3))
display(df[['district', 'price_billion_vnd', 'area_m2', 'bedrooms']].describe(include='all'))

Số bản ghi: 100


,property_id,title,district,ward,location_raw,price_million_vnd,price_billion_vnd,area_m2,price_per_m2_million,bedrooms,...,direction,position,latitude,longitude,description_snippet,distance_to_nearest_school_m,distance_to_nearest_park_m,distance_to_nearest_hospital_m,distance_to_nearest_supermarket_m,distance_to_nearest_boulevard_m
0,GV_001,"🔥 Giảm 440tr Nhà Quang Trung P10 GV, 20m2, 3.3x6m",Gò Vấp,Phường Gò Vấp,"Phường Gò Vấp,TP. Hồ Chí Minh(Mới)",1250.0,1.25,20.0,62.50,2,...,Nam,Trong hẻm,10.826903,106.673948,"🔥 Giảm 440tr Nhà Quang Trung P10 GV, 20m2, 3.3...",None,None,None,None,None
1,GV_002,⭐️ Giảm 50tr! Nhà Vuông SHR 2 tầng 26m2 – Quan...,Gò Vấp,Phường Thông Tây Hội,"Đường Quang Trung,Phường Thông Tây Hội,TP. Hồ ...",2850.0,2.85,26.0,109.62,2,...,NaN,Trong hẻm,10.842153,106.645882,⭐️ Giảm 50tr! Nhà Vuông SHR 2 tầng 26m2 – Quan...,None,None,None,None,None
2,GV_003,"Nhà 2 tầng sàn 52m2, Vuông – Lê Đức Thọ QV",Gò Vấp,Phường Gò Vấp,"Đường Lê Đức Thọ,Phường Gò Vấp,TP. Hồ Chí Mi...",3100.0,3.10,25.0,124.00,2,...,NaN,Đường chính,10.844656,106.674622,"Nhà 2 tầng sàn 52m2, Vuông – Lê Đức Thọ QV🏡 25...",None,None,None,None,None


,district,price_billion_vnd,area_m2,bedrooms
count,100,100.000000,100.00000,100.000000
unique,2,NaN,NaN,NaN
top,Gò Vấp,NaN,NaN,NaN
freq,50,NaN,NaN,NaN
mean,NaN,8.273000,69.10500,3.480000
std,NaN,4.103682,35.80164,1.445858
min,NaN,1.250000,20.00000,1.000000
25%,NaN,5.237500,44.45000,2.000000
50%,NaN,7.450000,62.50000,3.000000
75%,NaN,10.400000,83.50000,4.000000


## 1. Thiết Kế Truy Vấn

Mỗi BĐS sẽ gọi `1 request` tới Overpass và lấy về tất cả POI liên quan trong `MAX_RADIUS_M`.

Sau đó notebook sẽ tự map tag OSM thành các nhóm feature:
- `school`
- `park`
- `hospital`
- `supermarket`
- `market`
- `cafe`
- `boulevard` (đường lớn, xấp xỉ bằng highway chính)

## Bước 1. Xây Dựng Hàm Hỗ Trợ Và Hàm Gọi API

Mục đích:
- Tạo các hàm tính khoảng cách Haversine.
- Xây dựng câu truy vấn Overpass.
- Gọi API có cache, retry và timeout.
- Chuẩn hóa kết quả OSM thành các nhóm feature của project.

Input:
- `latitude`, `longitude` của từng bất động sản.
- Cấu hình bán kính tìm kiếm và category.

Output:
- Các hàm như `request_overpass(...)`, `normalize_osm_results(...)`, `build_feature_block(...)` để tái sử dụng cho toàn bộ pipeline.


In [4]:
def haversine_m(lat1, lon1, lat2, lon2):
    earth_radius_m = 6371000
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_m * c


def build_cache_key(lat, lon, radius_m):
    raw = f'{lat:.6f}|{lon:.6f}|{radius_m}'
    return hashlib.md5(raw.encode('utf-8')).hexdigest()


def overpass_query(lat, lon, radius_m):
    return f'''
[out:json][timeout:25];
(
  nwr["amenity"="school"](around:{radius_m},{lat},{lon});
  nwr["amenity"="kindergarten"](around:{radius_m},{lat},{lon});
  nwr["leisure"="park"](around:{radius_m},{lat},{lon});
  nwr["amenity"="hospital"](around:{radius_m},{lat},{lon});
  nwr["amenity"="clinic"](around:{radius_m},{lat},{lon});
  nwr["shop"="supermarket"](around:{radius_m},{lat},{lon});
  nwr["amenity"="marketplace"](around:{radius_m},{lat},{lon});
  nwr["amenity"="cafe"](around:{radius_m},{lat},{lon});
  way["highway"~"trunk|primary|secondary|trunk_link|primary_link|secondary_link"](around:{radius_m},{lat},{lon});
);
out center tags;
'''.strip()


def fetch_overpass_status_text():
    req = urllib.request.Request(
        OVERPASS_STATUS_URL,
        headers={'User-Agent': USER_AGENT},
        method='GET',
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT_SECONDS) as resp:
        return resp.read().decode('utf-8', errors='replace')


def parse_overpass_status(text):
    info = {
        'connected_as': None,
        'rate_limit': None,
        'slots_available_now': None,
        'raw_text': text,
    }

    match = re.search(r'Connected as:\s*(\S+)', text)
    if match:
        info['connected_as'] = match.group(1)

    match = re.search(r'Rate limit:\s*(\d+)', text)
    if match:
        info['rate_limit'] = int(match.group(1))

    match = re.search(r'(\d+)\s+slots available now\.', text)
    if match:
        info['slots_available_now'] = int(match.group(1))

    return info


def wait_for_overpass_slot():
    while True:
        status_text = fetch_overpass_status_text()
        status = parse_overpass_status(status_text)
        slots = status.get('slots_available_now')
        if slots is None or slots > 0:
            return status

        print(f'Chưa có slot khả dụng. Chờ thêm {STATUS_POLL_SECONDS}s rồi kiểm tra lại /api/status...')
        time.sleep(STATUS_POLL_SECONDS)


def request_overpass(lat, lon, radius_m=MAX_RADIUS_M, force_refresh=False):
    cache_key = build_cache_key(lat, lon, radius_m)
    cache_path = CACHE_DIR / f'{cache_key}.json'

    if cache_path.exists() and not force_refresh:
        with cache_path.open('r', encoding='utf-8') as f:
            return json.load(f)

    payload = urllib.parse.urlencode({'data': overpass_query(lat, lon, radius_m)}).encode('utf-8')
    req = urllib.request.Request(
        OVERPASS_URL,
        data=payload,
        headers={'User-Agent': USER_AGENT, 'Content-Type': 'application/x-www-form-urlencoded'},
        method='POST',
    )

    last_error = None
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            wait_for_overpass_slot()
            with urllib.request.urlopen(req, timeout=TIMEOUT_SECONDS) as resp:
                data = json.loads(resp.read().decode('utf-8'))
            with cache_path.open('w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            time.sleep(BASE_SLEEP_SECONDS)
            return data
        except urllib.error.HTTPError as exc:
            last_error = exc
            if exc.code == 429:
                wait_s = max(STATUS_POLL_SECONDS, 20)
                print(f'Bị 429 tại ({lat}, {lon}). Chờ {wait_s}s rồi thử lại...')
                time.sleep(wait_s)
                continue
            if exc.code == 504:
                wait_s = attempt * 20
                print(f'Bị 504 tại ({lat}, {lon}). Chờ {wait_s}s rồi thử lại...')
                time.sleep(wait_s)
                continue
            wait_s = attempt * 5
            print(f'Lần thử {attempt} thất bại cho ({lat}, {lon}): {exc}. Thử lại sau {wait_s}s')
            time.sleep(wait_s)
        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            wait_s = attempt * 10
            print(f'Lần thử {attempt} thất bại cho ({lat}, {lon}): {exc}. Thử lại sau {wait_s}s')
            time.sleep(wait_s)

    raise RuntimeError(f'Overpass request failed after {RETRY_COUNT} attempts: {last_error}')


def extract_point(element):
    if 'lat' in element and 'lon' in element:
        return element['lat'], element['lon']
    center = element.get('center')
    if center and 'lat' in center and 'lon' in center:
        return center['lat'], center['lon']
    return None, None


def classify_element(tags):
    amenity = tags.get('amenity')
    leisure = tags.get('leisure')
    shop = tags.get('shop')
    highway = tags.get('highway')

    if amenity in {'school', 'kindergarten'}:
        return 'school'
    if leisure == 'park':
        return 'park'
    if amenity in {'hospital', 'clinic'}:
        return 'hospital'
    if shop == 'supermarket':
        return 'supermarket'
    if amenity == 'marketplace':
        return 'market'
    if amenity == 'cafe':
        return 'cafe'
    if highway in {'trunk', 'primary', 'secondary', 'trunk_link', 'primary_link', 'secondary_link'}:
        return 'boulevard'
    return None


def normalize_osm_results(raw, property_lat, property_lon):
    grouped = {category: [] for category in CATEGORY_ORDER}

    for element in raw.get('elements', []):
        tags = element.get('tags', {})
        category = classify_element(tags)
        if not category:
            continue

        lat, lon = extract_point(element)
        if lat is None or lon is None:
            continue

        grouped[category].append({
            'name': tags.get('name', f'osm_{element.get("id", "unknown")}'),
            'lat': lat,
            'lon': lon,
            'distance_m': round(haversine_m(property_lat, property_lon, lat, lon)),
            'osm_type': element.get('type'),
            'osm_id': element.get('id'),
            'tags': tags,
        })

    for category in grouped:
        grouped[category].sort(key=lambda item: item['distance_m'])

    return grouped


def build_feature_block(grouped, count_radius_m=COUNT_RADIUS_M):
    features = {}

    for category in CATEGORY_ORDER:
        items = grouped.get(category, [])
        if items:
            features[f'distance_to_nearest_{category}_m'] = items[0]['distance_m']
            features[f'nearest_{category}_name'] = items[0]['name']
            features[f'near_{category}_count_1km'] = sum(item['distance_m'] <= count_radius_m for item in items)
        else:
            features[f'distance_to_nearest_{category}_m'] = None
            features[f'nearest_{category}_name'] = None
            features[f'near_{category}_count_1km'] = 0

    return features

## Bước 1.1. Thử Một Lần Gọi API Mẫu

Mục đích:
- Kiểm tra nhanh xem Overpass API có trả dữ liệu đúng không trước khi chạy toàn bộ 100 căn.
- Quan sát số lượng phần tử API trả về và bộ feature đầu ra.

Input:
- Một bất động sản mẫu từ `properties[0]`.

Output:
- `sample_raw`: raw response từ Overpass.
- `sample_grouped`: dữ liệu đã nhóm theo category.
- `sample_features`: bộ feature enrich của căn mẫu.


In [5]:
sample_prop = properties[0]
sample_raw = request_overpass(sample_prop['latitude'], sample_prop['longitude'])
sample_grouped = normalize_osm_results(sample_raw, sample_prop['latitude'], sample_prop['longitude'])
sample_features = build_feature_block(sample_grouped)

print(sample_prop['property_id'], sample_prop['district'])
print('Số phần tử từ API:', len(sample_raw.get('elements', [])))
sample_features

GV_001 Gò Vấp
Số phần tử từ API: 228


{'distance_to_nearest_school_m': 219,
 'nearest_school_name': 'Trường Trung học cơ sở&Trung học phổ thông Hồng Hà',
 'near_school_count_1km': 3,
 'distance_to_nearest_park_m': 597,
 'nearest_park_name': 'osm_446052467',
 'near_park_count_1km': 9,
 'distance_to_nearest_hospital_m': 1204,
 'nearest_hospital_name': 'Bệnh viện Quân y 175',
 'near_hospital_count_1km': 0,
 'distance_to_nearest_supermarket_m': 934,
 'nearest_supermarket_name': 'Co.opmart Phan Văn Trị',
 'near_supermarket_count_1km': 1,
 'distance_to_nearest_market_m': 1498,
 'nearest_market_name': 'Chợ Gò Vấp',
 'near_market_count_1km': 0,
 'distance_to_nearest_cafe_m': 303,
 'nearest_cafe_name': 'Epatta Coffee',
 'near_cafe_count_1km': 11,
 'distance_to_nearest_boulevard_m': 171,
 'nearest_boulevard_name': 'osm_226512045',
 'near_boulevard_count_1km': 73}

## 2. Chạy Toàn Bộ Quá Trình Enrich

Notebook này không nên bắn toàn bộ 100 request liên tục vào public Overpass.

Cách chạy an toàn hơn:
- chạy theo batch nhỏ, mặc định `15` căn mỗi lượt
- nghỉ giữa các batch để giảm nguy cơ `429` hoặc `504`
- đọc cache và checkpoint nếu đã có kết quả trước đó
- kiểm tra `api/status` trước khi gọi
- nếu gặp `429` hoặc `504` thì chờ rồi thử lại
- nếu vẫn lỗi thì ghi log và chuyển sang căn tiếp theo nếu `CONTINUE_ON_ERROR = True`
- tiếp tục loop cho đến khi đủ `TARGET_TOTAL_PROPERTIES` hoặc hết dữ liệu mục tiêu

Input:
- Danh sách `properties` từ file JSON gốc.
- Các hàm gọi API và chuẩn hóa dữ liệu đã định nghĩa ở bước trước.
- Cấu hình `START_INDEX`, `TARGET_TOTAL_PROPERTIES`, `MAX_PROPERTIES_PER_RUN`, `RUN_UNTIL_DONE`, `BATCH_PAUSE_SECONDS`.

Output:
- Danh sách `enriched_properties`, trong đó mỗi bản ghi đã có thêm các field như `distance_to_nearest_*`, `nearest_*_name`, `near_*_count_1km`, `api_result_count`.
- File checkpoint tạm, file output cuối và file log lỗi nếu có.


In [6]:
def enrich_property_with_overpass(prop, force_refresh=False):
    lat = prop['latitude']
    lon = prop['longitude']
    raw = request_overpass(lat, lon, radius_m=MAX_RADIUS_M, force_refresh=force_refresh)
    grouped = normalize_osm_results(raw, lat, lon)
    features = build_feature_block(grouped, count_radius_m=COUNT_RADIUS_M)

    enriched = dict(prop)
    enriched.update(features)
    enriched['enrichment_provider'] = 'osm_overpass_api'
    enriched['enrichment_radius_m'] = MAX_RADIUS_M
    enriched['enrichment_count_radius_m'] = COUNT_RADIUS_M
    enriched['api_result_count'] = len(raw.get('elements', []))
    return enriched


if CHECKPOINT_PATH.exists() and not FORCE_REFRESH:
    with CHECKPOINT_PATH.open('r', encoding='utf-8') as f:
        enriched_properties = json.load(f)
    print(f'Đã nạp checkpoint có sẵn với {len(enriched_properties)} bản ghi')
else:
    enriched_properties = []

if ERROR_LOG_PATH.exists() and not FORCE_REFRESH:
    with ERROR_LOG_PATH.open('r', encoding='utf-8') as f:
        error_logs = json.load(f)
else:
    error_logs = []

all_target_properties = properties[START_INDEX: START_INDEX + TARGET_TOTAL_PROPERTIES]
target_property_ids = {prop['property_id'] for prop in all_target_properties}
existing_ids = {item['property_id'] for item in enriched_properties if item['property_id'] in target_property_ids}
batch_counter = 0

print(f'Mục tiêu enrich: {len(all_target_properties)} bất động sản')
print(f'Đã có sẵn trong checkpoint: {len(existing_ids)} bất động sản')

while True:
    remaining_properties = [
        prop for prop in all_target_properties
        if FORCE_REFRESH or prop['property_id'] not in existing_ids
    ]

    if not remaining_properties:
        print('Không còn bất động sản nào cần enrich. Đã hoàn tất toàn bộ phạm vi mục tiêu.')
        break

    if len(existing_ids) >= len(target_property_ids):
        print('Đã đủ số lượng BĐS mục tiêu.')
        break

    if not RUN_UNTIL_DONE and batch_counter >= 1:
        print('RUN_UNTIL_DONE = False nên dừng sau 1 batch.')
        break

    if batch_counter >= MAX_BATCHES:
        print(f'Đã chạm giới hạn MAX_BATCHES = {MAX_BATCHES}. Dừng để tránh chạy quá lâu.')
        break

    current_batch = remaining_properties[:MAX_PROPERTIES_PER_RUN]
    batch_counter += 1
    processed_in_this_batch = 0

    print(f'\n=== Batch {batch_counter} ===')
    print(f'Số bản ghi còn lại trong mục tiêu: {len(remaining_properties)}')
    print(f'Batch này sẽ xử lý {len(current_batch)} căn')
    print('Danh sách property_id:', [prop['property_id'] for prop in current_batch])

    for prop in current_batch:
        try:
            enriched = enrich_property_with_overpass(prop, force_refresh=FORCE_REFRESH)
            # Nếu đã có bản ghi cũ cùng property_id thì thay thế để tránh trùng
            enriched_properties = [item for item in enriched_properties if item['property_id'] != prop['property_id']]
            enriched_properties.append(enriched)
            existing_ids.add(prop['property_id'])
            processed_in_this_batch += 1
            print(f"Xong {prop['property_id']} | API elements: {enriched['api_result_count']}")

            if processed_in_this_batch % SAVE_CHECKPOINT_EVERY == 0:
                snapshot = sorted(enriched_properties, key=lambda x: x['property_id'])
                with CHECKPOINT_PATH.open('w', encoding='utf-8') as f:
                    json.dump(snapshot, f, ensure_ascii=False, indent=2)
                with OUTPUT_PATH.open('w', encoding='utf-8') as f:
                    json.dump(snapshot, f, ensure_ascii=False, indent=2)
                print(f'Đã lưu checkpoint/output với {len(existing_ids)} / {len(target_property_ids)} bản ghi mục tiêu')

        except Exception as exc:
            error_logs.append({
                'property_id': prop['property_id'],
                'district': prop['district'],
                'latitude': prop['latitude'],
                'longitude': prop['longitude'],
                'error': str(exc),
                'batch': batch_counter,
            })
            print(f"Lỗi tại {prop['property_id']}: {exc}")
            if not CONTINUE_ON_ERROR:
                raise

    enriched_properties = sorted(enriched_properties, key=lambda x: x['property_id'])
    with CHECKPOINT_PATH.open('w', encoding='utf-8') as f:
        json.dump(enriched_properties, f, ensure_ascii=False, indent=2)

    with OUTPUT_PATH.open('w', encoding='utf-8') as f:
        json.dump(enriched_properties, f, ensure_ascii=False, indent=2)

    with ERROR_LOG_PATH.open('w', encoding='utf-8') as f:
        json.dump(error_logs, f, ensure_ascii=False, indent=2)

    print(f'Hoàn tất batch {batch_counter}')
    print('Số bản ghi enrich trong mục tiêu hiện có:', len(existing_ids))
    print('Tổng số lỗi tích lũy:', len(error_logs))

    remaining_after_batch = [
        prop for prop in all_target_properties
        if FORCE_REFRESH or prop['property_id'] not in existing_ids
    ]
    if not remaining_after_batch:
        print('Đã enrich xong toàn bộ các căn trong phạm vi mục tiêu.')
        break

    if RUN_UNTIL_DONE:
        print(f'Nghỉ {BATCH_PAUSE_SECONDS}s trước khi chạy batch tiếp theo...')
        time.sleep(BATCH_PAUSE_SECONDS)
    else:
        break

enriched_properties = sorted(
    [item for item in enriched_properties if item['property_id'] in target_property_ids],
    key=lambda x: x['property_id']
)
with OUTPUT_PATH.open('w', encoding='utf-8') as f:
    json.dump(enriched_properties, f, ensure_ascii=False, indent=2)

with ERROR_LOG_PATH.open('w', encoding='utf-8') as f:
    json.dump(error_logs, f, ensure_ascii=False, indent=2)

print('\nHoàn tất toàn bộ lượt chạy notebook')
print('Tổng số bản ghi enrich trong mục tiêu:', len(enriched_properties))
print(f'File output hiện tại đã được đồng bộ tại: {OUTPUT_PATH}')
print('Tổng số lỗi:', len(error_logs))

Chạy từ index 0 đến 14 trên tổng số 100 bất động sản
Xong GV_001 | API elements: 228
Xong GV_002 | API elements: 102
Xong GV_003 | API elements: 132
Xong GV_004 | API elements: 254
Xong GV_005 | API elements: 112
Đã lưu checkpoint với 5 bản ghi
Xong GV_006 | API elements: 163
Xong GV_007 | API elements: 163
Xong GV_008 | API elements: 144
Xong GV_009 | API elements: 173


Bị 504 tại (10.842521, 106.663364). Chờ 20s rồi thử lại...


Xong GV_010 | API elements: 181
Đã lưu checkpoint với 10 bản ghi


Xong GV_011 | API elements: 188


Bị 429 tại (10.846264351707, 106.66354684903). Chờ 20s rồi thử lại...


Xong GV_012 | API elements: 169


Xong GV_013 | API elements: 74


Xong GV_014 | API elements: 179


Xong GV_015 | API elements: 128
Đã lưu checkpoint với 15 bản ghi
Hoàn tất lượt chạy này
Số bản ghi enrich hiện có: 15
Số lỗi trong lượt chạy: 0


## Bước 2.1. Xem Nhanh Kết Quả Enrich

Mục đích:
- Chuyển output enrich sang `DataFrame` để xem nhanh vài cột quan trọng.
- Xác nhận các field mới đã được sinh đúng schema mong muốn.

Input:
- `enriched_properties`.

Output:
- `enriched_df` và bảng preview các cột khoảng cách, số lượng POI, số phần tử API trả về.


In [7]:
enriched_df = pd.DataFrame(enriched_properties)
preview_cols = [
    'property_id',
    'district',
    'distance_to_nearest_school_m',
    'distance_to_nearest_park_m',
    'distance_to_nearest_hospital_m',
    'distance_to_nearest_supermarket_m',
    'distance_to_nearest_market_m',
    'distance_to_nearest_cafe_m',
    'distance_to_nearest_boulevard_m',
    'api_result_count',
]
display(enriched_df[preview_cols].head(10))

,property_id,district,distance_to_nearest_school_m,distance_to_nearest_park_m,distance_to_nearest_hospital_m,distance_to_nearest_supermarket_m,distance_to_nearest_market_m,distance_to_nearest_cafe_m,distance_to_nearest_boulevard_m,api_result_count
0,GV_001,Gò Vấp,219,597,1204,934,1498,303,171,228
1,GV_002,Gò Vấp,397,425,839,240,143,275,243,102
2,GV_003,Gò Vấp,447,294,321,1343,479,565,144,132
3,GV_004,Gò Vấp,324,427,1058,727,1320,275,139,254
4,GV_005,Gò Vấp,408,349,485,582,778,534,359,112
5,GV_006,Gò Vấp,442,375,1082,721,1401,351,240,163
6,GV_007,Gò Vấp,442,375,1082,721,1401,351,240,163
7,GV_008,Gò Vấp,80,745,235,560,457,309,105,144
8,GV_009,Gò Vấp,272,413,692,1116,875,328,322,173
9,GV_010,Gò Vấp,224,603,905,992,847,188,194,181


## 3. Kiểm Tra Chất Lượng

Mục đích:
- Kiểm tra phân bố các cột khoảng cách và số lượng POI.
- So sánh nhanh giữa Gò Vấp và Tân Bình.
- Phát hiện category nào đang bị thiếu dữ liệu nhiều.

Input:
- `enriched_df` sau khi chạy API enrich.

Output:
- Bảng thống kê mô tả cho distance/count columns.
- Bảng trung bình theo quận.
- Bảng tổng hợp giá trị thiếu.


In [8]:
distance_cols = [col for col in enriched_df.columns if col.startswith('distance_to_nearest_')]
count_cols = [col for col in enriched_df.columns if col.startswith('near_') and col.endswith('_count_1km')]

display(enriched_df[distance_cols].describe().T)
display(enriched_df[count_cols].describe().T)
display(enriched_df.groupby('district')[distance_cols].mean().round(1))

,count,mean,std,min,25%,50%,75%,max
distance_to_nearest_school_m,15.0,280.800000,134.011833,80.0,159.5,272.0,402.5,447.0
distance_to_nearest_park_m,15.0,564.333333,238.298033,294.0,394.0,479.0,674.0,1105.0
distance_to_nearest_hospital_m,15.0,805.200000,322.128014,235.0,588.5,839.0,1070.0,1275.0
distance_to_nearest_supermarket_m,15.0,808.666667,356.960916,55.0,651.5,746.0,1087.0,1343.0
distance_to_nearest_boulevard_m,15.0,242.466667,136.831944,82.0,141.5,240.0,324.0,618.0
distance_to_nearest_market_m,15.0,835.800000,438.152159,143.0,468.0,778.0,1259.5,1498.0
distance_to_nearest_cafe_m,15.0,337.533333,145.595755,21.0,275.0,312.0,372.0,592.0


,count,mean,std,min,25%,50%,75%,max
near_school_count_1km,15.0,9.466667,6.162869,3.0,3.5,8.0,12.5,20.0
near_park_count_1km,15.0,4.466667,2.722044,0.0,3.0,4.0,5.5,10.0
near_hospital_count_1km,15.0,1.133333,0.990430,0.0,0.0,1.0,2.0,3.0
near_supermarket_count_1km,15.0,0.800000,0.676123,0.0,0.0,1.0,1.0,2.0
near_market_count_1km,15.0,0.866667,0.743223,0.0,0.0,1.0,1.0,2.0
near_cafe_count_1km,15.0,11.000000,5.694609,1.0,6.5,11.0,15.5,19.0
near_boulevard_count_1km,15.0,36.200000,20.114671,3.0,30.0,33.0,44.0,78.0


,distance_to_nearest_school_m,distance_to_nearest_park_m,distance_to_nearest_hospital_m,distance_to_nearest_supermarket_m,distance_to_nearest_boulevard_m,distance_to_nearest_market_m,distance_to_nearest_cafe_m
district,,,,,,,
Gò Vấp,280.8,564.3,805.2,808.7,242.5,835.8,337.5


In [9]:
missing_summary = enriched_df[distance_cols].isna().sum().sort_values(ascending=False)
missing_summary

distance_to_nearest_school_m         0
distance_to_nearest_park_m           0
distance_to_nearest_hospital_m       0
distance_to_nearest_supermarket_m    0
distance_to_nearest_boulevard_m      0
distance_to_nearest_market_m         0
distance_to_nearest_cafe_m           0
dtype: int64

## 4. Lưu Kết Quả

Mục đích:
- Ghi dữ liệu đã enrich ra file JSON để các pipeline khác dùng lại. Ở phiên bản hiện tại, file output cũng được cập nhật lại sau mỗi batch để tránh lệch với checkpoint.

Input:
- `enriched_properties`.
- Đường dẫn `OUTPUT_PATH`.

Output:
- File `data/overpass/go_vap_tan_binh_100_enriched_overpass_api.json`.


In [10]:
with OUTPUT_PATH.open('w', encoding='utf-8') as f:
    json.dump(enriched_properties, f, ensure_ascii=False, indent=2)

print(f'Saved {len(enriched_properties)} records to {OUTPUT_PATH}')

Saved 15 records to /Users/totien/Documents/caohoc/Ki2-2026/IT2041_G8_DecisionMaking/data/go_vap_tan_binh_100_enriched_overpass_api.json


## 5. Gợi Ý Tích Hợp Vào Project

Để pipeline này dùng tốt trong project, nên handle theo cách sau:

1. Chạy enrichment theo batch `15` căn rồi nghỉ giữa các batch, thay vì bắn toàn bộ 100 căn trong một lần.
2. Dùng checkpoint để nếu notebook dừng giữa chừng vẫn có thể resume và chạy tiếp cho đủ 100 căn.
3. Commit code, không commit cache raw API nếu cache quá lớn; ưu tiên giữ cache và output trong `data/overpass/` để tách biệt với dữ liệu gốc.
4. Giữ output schema ổn định để solution 1, solution 2 và validation dùng chung.
5. Khi dùng public Overpass, nên theo dõi `https://overpass-api.de/api/status` để xem số slot hiện có.
6. Theo tài liệu chính thức, public instance phù hợp cho nhu cầu nhỏ và có guideline khoảng `10.000 request/ngày`, dưới `1 GB/ngày`, nhưng quota thực tế còn phụ thuộc tải hệ thống và thời gian chạy truy vấn.
7. Nếu cần enrich đủ 100 căn nhanh và ổn định hơn, nên chuyển sang provider thương mại như `Google Places` hoặc `Geoapify`, hoặc tự dựng nguồn OSM offline / private instance.
8. Nếu vẫn dùng Overpass, nên tận dụng cache và tránh gọi lại những căn đã enrich xong.